In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!nvidia-smi

Thu Apr 30 02:20:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -U transformers accelerate peft datasets
!pip install finnhub-python
!pip install -U bitsandbytes>=0.46.1
import os
import re
import csv
import math
import time
import json
import random
import finnhub
import datasets
import pandas as pd
import yfinance as yf
from datetime import datetime
from collections import defaultdict
from datasets import Dataset
from openai import OpenAI
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig # Added BitsAndBytesConfig import
)
from peft import LoraConfig, get_peft_model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 66.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 95.3 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.3
    Uninstalling datasets-4.8.3:
      Successfully uninstalled datasets-4.8.3
  Att

In [4]:
from datasets import load_dataset
ds = load_dataset("FinGPT/fingpt-forecaster-dow30-202305-202405")
test_df = ds["test"].to_pandas()
test_samples = test_df.sample(20, random_state=42)
print(f"Test samples: {len(test_samples)}")
print(test_samples[['symbol','label']].value_counts())

README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00001-7c4c80aa07272d(…):   0%|          | 0.00/3.57M [00:00<?, ?B/s]

(…)-00000-of-00001-28531804b005ddc6.parquet:   0%|          | 0.00/925k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1230 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Test samples: 20
symbol  label               
AXP     down by 0-1%            1
        down by 1-2%            1
CAT     up by 3-4%              1
CRM     down by more than 5%    1
        up by 4-5%              1
CSCO    down by 0-1%            1
DIS     down by 1-2%            1
IBM     down by 0-1%            1
JPM     up by 4-5%              1
MCD     down by 4-5%            1
MMM     up by 2-3%              1
MRK     down by 3-4%            1
MSFT    up by 2-3%              1
NKE     up by 3-4%              1
PG      up by 0-1%              1
TRV     down by 0-1%            1
        up by 0-1%              1
UNH     down by 2-3%            1
        down by 3-4%            1
WBA     down by more than 5%    1
Name: count, dtype: int64


In [5]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")
!nvidia-smi

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.
Thu Apr 30 02:23:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             28W /   70W |     621MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------

In [11]:
def extract_news_core(prompt_text: str) -> str:
    
    headlines = re.findall(r'\[Headline\]:\s*(.+)', prompt_text)
    summaries = re.findall(r'\[Summary\]:\s*(.+)', prompt_text)
    news_lines = []
    for h, s in zip(headlines, summaries):
        news_lines.append(f"- {h.strip()} | {s.strip()[:80]}")
    return '\n'.join(news_lines[:8])


def build_prompt(news_text: str) -> str:
    news_core = extract_news_core(news_text)
    system = (
        "You are a financial analyst. "
        "First briefly analyze the news sentiment in one sentence, "
        "then output a JSON object on the last line. "
        'JSON format: {"sentiment_score": <integer -2 to 2>, '
        '"direction": "<Bullish|Neutral|Bearish>", '
        '"confidence": <float 0.0 to 1.0>}'
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": f"Analyze:\n{news_core}"}
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def keyword_fallback(news_text: str) -> dict:

    text = news_text.lower()
    pos_words = ['growth','beat','positive','surge','dividend',
                 'increase','record','profit','strong','raised']
    neg_words = ['concern','drop','decline','risk','miss','fell',
                 'loss','warn','cut','weak','below']
    score = 0.0
    for w in pos_words:
        if w in text: score += 0.4
    for w in neg_words:
        if w in text: score -= 0.4
    score = max(min(score, 2.0), -2.0)
    direction = "Bullish" if score > 0 else ("Bearish" if score < 0 else "Neutral")
    return {
        "sentiment_score": round(score, 1),
        "direction": direction,
        "confidence": 0.3,
        "note": "keyword_fallback"
    }


def get_signal(news_text: str, model, tokenizer) -> dict:
    prompt = build_prompt(news_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    all_json = re.findall(r'\{.*?\}', response, re.DOTALL)
    if all_json:
        try:
            data = json.loads(all_json[-1])
            if "sentiment_score" in data and "direction" in data:
                data["sentiment_score"] = max(-2, min(2, int(data["sentiment_score"])))
                data["confidence"] = max(0.0, min(1.0, float(data["confidence"])))
                data["note"] = "llm_output"
                data["reasoning"] = response.split('{')[0].strip()[:120]
                return data
        except (json.JSONDecodeError, KeyError, ValueError):
            pass

    return keyword_fallback(news_text)

In [13]:
results = []
for index, row in test_samples.iterrows():
    sig = get_signal(str(row['prompt']), model, tokenizer)
    sig['symbol']   = row.get('symbol', '')
    sig['gt_label'] = row.get('label', '')
    results.append(sig)
    print(f"[{row.get('symbol',''):4s}] {sig['direction']:8s} | "
          f"score={float(sig['sentiment_score']):+.1f} | "  
          f"conf={sig['confidence']:.2f} | "
          f"via={sig['note']:16s} | "
          f"gt={str(row.get('label',''))[:15]}")

final_df = pd.DataFrame(results)
final_df.to_csv("signals.csv", index=False)

[PG  ] Neutral  | score=+0.0 | conf=0.70 | via=llm_output       | gt=up by 0-1%
[WBA ] Neutral  | score=+0.0 | conf=0.75 | via=llm_output       | gt=down by more th
[MCD ] Bearish  | score=-1.0 | conf=0.80 | via=llm_output       | gt=down by 4-5%
[AXP ] Bearish  | score=+0.0 | conf=0.80 | via=llm_output       | gt=down by 0-1%
[CRM ] Bearish  | score=+0.0 | conf=0.64 | via=llm_output       | gt=up by 4-5%
[UNH ] Bearish  | score=-1.0 | conf=0.89 | via=llm_output       | gt=down by 3-4%
[NKE ] Bearish  | score=+0.0 | conf=0.60 | via=llm_output       | gt=up by 3-4%
[IBM ] Bearish  | score=+0.0 | conf=0.56 | via=llm_output       | gt=down by 0-1%
[AXP ] Bullish  | score=+0.0 | conf=0.95 | via=llm_output       | gt=down by 1-2%
[MRK ] Bearish  | score=+0.0 | conf=0.60 | via=llm_output       | gt=down by 3-4%
[CRM ] Neutral  | score=+0.0 | conf=0.70 | via=llm_output       | gt=down by more th
[CSCO] Neutral  | score=+0.0 | conf=0.60 | via=llm_output       | gt=down by 0-1%
[TRV ] Bullish  

In [14]:
print("="*55)
print("RESULTS SUMMARY")
print("="*55)
print(f"Total samples       : {len(final_df)}")
print(f"LLM success rate    : {(final_df['note']=='llm_output').sum()}/{len(final_df)}")
print(f"Mean confidence     : {final_df['confidence'].mean():.3f}")
print(f"\nPredicted direction distribution:")
print(final_df['direction'].value_counts().to_string())
print(f"\nGround truth label distribution:")
print(final_df['gt_label'].value_counts().to_string())


def gt_to_direction(label):
    label = str(label).lower()
    if 'up' in label:   return 'Bullish'
    if 'down' in label: return 'Bearish'
    return 'Neutral'

final_df['gt_direction'] = final_df['gt_label'].apply(gt_to_direction)
accuracy = (final_df['direction'] == final_df['gt_direction']).mean()
print(f"\nDirection accuracy  : {accuracy:.1%}")
print("="*55)

RESULTS SUMMARY
Total samples       : 20
LLM success rate    : 19/20
Mean confidence     : 0.694

Predicted direction distribution:
direction
Bearish    11
Neutral     6
Bullish     3

Ground truth label distribution:
gt_label
down by 0-1%            4
up by 0-1%              2
down by more than 5%    2
up by 4-5%              2
down by 3-4%            2
up by 3-4%              2
up by 2-3%              2
down by 1-2%            2
down by 4-5%            1
down by 2-3%            1

Direction accuracy  : 30.0%
